# Data Preprocessing

## Load dataset

In [ ]:
from datasets import load_dataset


# Dataset 1 — wikidoc (~10k samples)
wikidoc = load_dataset("medalpaca/medical_meadow_wikidoc", split="train")

# Dataset 2 — PubMedQA (~200k samples)
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc.json:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/233M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211269 [00:00<?, ? examples/s]

In [ ]:
print(wikidoc)
print(pubmedqa)

Dataset({
    features: ['input', 'output', 'instruction'],
    num_rows: 10000
})
Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
    num_rows: 211269
})


# Combine datasets

In [ ]:
# ghép cho wikidoc (lay output)
def extract_wikidoc(example):
    text = f"{example['output']}"
    return {"text": text}

# ghép cho pubmedqa (context + long_answer)
def extract_pubmed(example):
    # vi contexts la list cac doan -> join lai thanh 1 doan cach nhau bang dau cach
    contexts = " ".join(example["context"]["contexts"])
    long_answer = example["long_answer"]

    text = f"{contexts}\n{long_answer}"
    return {"text": text}



wikidoc_text = wikidoc.map(extract_wikidoc, remove_columns=wikidoc.column_names)
pubmedqa_text = pubmedqa.map(extract_pubmed, remove_columns=pubmedqa.column_names)

print(wikidoc_text)
print(wikidoc_text[0])
print("---")
print(pubmedqa_text)
print(pubmedqa_text[0])


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/211269 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 10000
})
{'text': 'Squamous cell carcinoma of the lung may be classified according to the WHO histological classification system into 4 main types: papillary, clear cell, small cell, and basaloid.'}
---
Dataset({
    features: ['text'],
    num_rows: 211269
})
{'text': 'Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated. The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease. A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tu

In [ ]:
# concat 2 cái data lại với nhau
from datasets import concatenate_datasets

combined_datasets = concatenate_datasets([wikidoc_text, pubmedqa_text])

In [ ]:
print(combined_datasets)

Dataset({
    features: ['text'],
    num_rows: 221269
})


In [ ]:

# shuffle
SEED = 23
combined_datasets = combined_datasets.shuffle(seed=SEED)

# take small subset because it too large, take too much computation cost
combined_datasets = combined_datasets.select(range(5000))

print(combined_datasets[0])
print(combined_datasets[1])
print(combined_datasets)

{'text': 'Recent evidence suggests that CD4+CD25+FoxP3+ regulatory T-cells (Treg) may be responsible for the failure of host anti-tumour immunity by suppressing cytotoxic T- cells. We assessed the prognostic significance of tumour infiltrating lymphocytes (TIL) in intestinal-type gastric cardiac cancer. Tumour infiltrating lymphocyte (TIL) subsets and tumour infiltrating macrophages (TIM) were investigated in 52 cases using tissue microarrays. The interrelationship between the cell populations (CD3+, CD8+, CD20+, CD68+, GranzymeB+, FoxP3+) in different compartments and NED-survival was investigated (median follow-up time: 61 months). Intraepithelial infiltration with TIL and TIM including Treg was generally low and not related to NED-survival. However, patients with large numbers of FoxP3+ Treg in the tumour stroma (>125.9 FoxP3+TILs/mm2) had a median survival time of 58 months while those with low FoxP3+ TIL counts (<125.9 FoxP3+TILs/mm2) had a median survival time of 32 months (p = 0

## Tokenize

In [ ]:
# tokenize
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

def tokenize(examples): # examples lúc này là nguyên cái batch vì batch = true
    return tokenizer(
        examples["text"], # lấy cái list chứa batch của text
        # tokenizer của hugging face tự nhận 1 list -> trả về dict của list
        truncation=False, # cắt bớt nếu vượt quá số lượng max_token
        padding=False, # đệm padding
        # -> ko vì lát dùng packing
    )

tokenized = combined_datasets.map(
    tokenize, #
    batched=True, # tokenize nhiều sample cùng lúc
    # nếu batch = false thì example là 1 sample= {"text": "..."}
    # batch = true thì là dict of a list examples = {"text": ["...", "..."]}
    batch_size=1000,
    remove_columns=["text"],
    # loại cột text ra vì ko cần, vì map thì nó là tạo ra 2 cái column mới (với cái này là input_ids, attention_mask)
)

print(tokenized)
print(tokenized[0])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 10000
})
{'input_ids': [25140, 5904, 13230, 429, 11078, 19, 10, 6484, 17, 20, 10, 47314, 47, 18, 10, 22515, 350, 1786, 6436, 320, 51, 1580, 8, 1231, 387, 8480, 369, 279, 7901, 315, 3468, 7147, 2385, 372, 413, 39268, 553, 97695, 78809, 90576, 350, 12, 7761, 13, 1205, 31348, 279, 62803, 535, 292, 25361, 315, 15394, 413, 42264, 1095, 42645, 56778, 320, 51, 1715, 8, 304, 62800, 10604, 88285, 46245, 9387, 13, 350, 372, 413, 42264, 1095, 42645, 78659, 320, 51, 1715, 8, 74505, 323, 15394, 413, 42264, 1095, 18072, 759, 1134, 320, 34148, 8, 1033, 26219, 304, 220, 20, 17, 5048, 1667, 19847, 8003, 66893, 13, 576, 946, 36095, 1948, 279, 2779, 21910, 320, 6484, 18, 44662, 11078, 23, 44662, 11078, 17, 15, 44662, 11078, 21, 23, 44662, 2825, 12070, 30118, 33, 44662, 13282, 47, 18, 36197, 304, 2155, 86252, 323, 451, 1479, 67706, 85, 3936, 572, 26219, 320, 55651, 1795, 5239, 882, 25, 220, 21, 16, 3951, 568, 758, 2172, 747, 410, 58444

In [ ]:
# self test tokenizer
test_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
test = test_tokenizer(["hello", "BOYY"])
test.keys

<bound method Mapping.keys of {'input_ids': [[14990], [4677, 10060]], 'attention_mask': [[1], [1, 1]]}>

## Packing

In [ ]:
# PACKING
# ghép các sample lại với nhau để sao cho nó = đúng max token

def pack_tokens(examples, block_size=2048):
    # examples["input_ids"] là list of list


    # 1. concat tất cả token thành 1 list dài
    all_tokens = []
    for ids in examples["input_ids"]:
        all_tokens.extend(ids)
        # token EOS end of sentence de ngat giua cac sample
        all_tokens.append(tokenizer.eos_token_id)

    # 2. chia thành các chunks có đúng block_size
    chunks_input_ids = []
    chunks_attentions_mask = []

    # it like range(0, len() - 1, 1) but here 1 is blocksize
    for i in range(0, len(all_tokens) - block_size, block_size):
        chunk = all_tokens[i: i + block_size]
        chunks_input_ids.append(chunk) # list of list, each list is len of block-size
        chunks_attentions_mask.append([1] * block_size) # because al token are real token, not padding, and this mean assign 1 * blocksize

    return {
        "input_ids": chunks_input_ids,
        "attention_mask": chunks_attentions_mask,
    }

packed = tokenized.map(
    pack_tokens,
    batched=True,
    batch_size=5000,
    remove_columns=tokenized.column_names,
)

print(packed)
print(packed[0]["input_ids"][:10])
print(len(packed[0]["input_ids"]))


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 1833
})
[25140, 5904, 13230, 429, 11078, 19, 10, 6484, 17, 20]
2048


## Data Collator

In [ ]:
# tạo labels từ input_ids, nó thục hiện shift 1 token sang trái
# input_ids: [50, 98, 1234, 567, ...]
# labels:    [98, 1234, 567, ???, ...] shift 1 token sang trái

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, # mục đích để lấy tokenizer.pad_token_id
    # thục ra sau khi packing thì ko cần padding, nhưng đó là tham số bắt buộc nên vẫn phải truyền vào
    mlm=False, # causal LM, Không phải masked LM
)

# sự khác nhau giữa causal LM (GPT style) và masked LM (BERT style)
# masked che ngẫu nhiên, xong có thể nhin trái hoặc phải để predit , loss tính trên tokn bị mask
# causal là che hết phải, chỉ nhìn trái predict phải, loss ính trên toàn bộ token


## load Model  

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    torch_dtype=torch.bfloat16, # bfloat giu nguyen phan exponent cuar float32 chi thay doi man mantissa (precision) xuong chinh xac 4 chu so thay vi 8 chu so cuar float32, trach viec gradient qua lon gay Nan
)

total_params = sum(p.numel() for p in model.parameters())
print(f"total params: {total_params}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

total params: 494032768


## TrainingArguments

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="qwen-medical-pretrained",

    # batch_size
    per_device_train_batch_size=1, # 1 batch = 4 chunks
    gradient_accumulation_steps=32, #   1 step = 8 batches
    # bình thường thì hết 1 batch là update, còn cái này nó sẽ tích lũy lại rồi mới up

    # learning rate
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05, # 5% steps se tang lr tu 0 len 1e-4, trach update manh tu khi gradient chua on dinh


    # Duration
    num_train_epochs=1,

    # Save checkpoint
    save_steps=500, # steps số lần model update weight
    save_total_limit=3,

    # logging
    logging_steps=10,

    # Precision
    bf16=True,

    # Memory
    gradient_checkpointing=True,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Trainer

In [ ]:
from transformers import Trainer


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=packed,
    data_collator=data_collator,
)


trainer.train()

Step,Training Loss
10,2.145897
20,2.102024


In [ ]:
# save
trainer.save_model("qwen-medical-pretrained")
tokenizer.save_pretrained("qwen-medical-pretrained")